In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from setting_for_sdm.constants import CONSTANTS


# ============================================================
# CONFIG
# ============================================================

FILE_PATH = "./languages.csv"

# Main specification
MAIN_WINDOW = 8               # 8 quarters = 2 years
MAIN_ANNUAL_THRESHOLD = 0.05   # ±5% annual change

# ChatGPT release: 2022-11-30
# Quarterly data이므로 GPT 직전 마지막 완전한 분기
PRE_GPT_YEAR = 2022
PRE_GPT_QUARTER = 3

# 분석하고 싶은 특정 언어만 있다면 리스트 입력
# None이면 모든 programming language 사용
FOCAL_LANGUAGES =  [
        CONSTANTS.language_map[lang] for lang in CONSTANTS.languages_from2020to2022  if CONSTANTS.language_map[lang] is not None
    ]

In [2]:
len(FOCAL_LANGUAGES)

29

In [3]:


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(FILE_PATH)

print(df.shape)
print(df.columns)

# ============================================================
# 2. KEEP PROGRAMMING LANGUAGES ONLY
# ============================================================

df = df.loc[
    df["language_type"] == "programming"
].copy()


# ============================================================
# 3. AGGREGATE COUNTRY-LEVEL DATA
#
# 현재 데이터는 country × language × quarter 수준이므로
# country를 합쳐서 language × quarter 수준으로 변환
# ============================================================

lang_q = (
    df.groupby(
        ["year", "quarter", "language"],
        as_index=False
    )
    .agg(
        num_pushers=("num_pushers", "sum")
    )
)

(180338, 6)
Index(['num_pushers', 'language', 'language_type', 'iso2_code', 'year',
       'quarter'],
      dtype='object')


In [4]:
lang_q.head()

,year,quarter,language,num_pushers
0,2020,1,1C Enterprise,667
1,2020,1,AGS Script,640
2,2020,1,AMPL,1788
3,2020,1,ANTLR,7842
4,2020,1,ActionScript,2249


In [5]:

# ============================================================
# 4. CREATE TIME INDEX
# ============================================================

# 2020Q1 = 0이 되도록 sequential quarter index 생성

min_year = lang_q["year"].min()

lang_q["time"] = (
    (lang_q["year"] - min_year) * 4
    + (lang_q["quarter"] - 1)
)

lang_q["period"] = (
    lang_q["year"].astype(str)
    + "Q"
    + lang_q["quarter"].astype(str)
)


# ============================================================
# 5. COMPLETE LANGUAGE × QUARTER PANEL
#
# 특정 quarter에 언어가 아예 없으면
# num_pushers = 0으로 처리
# ============================================================

quarters = (
    lang_q[
        ["year", "quarter", "time", "period"]
    ]
    .drop_duplicates()
    .sort_values("time")
)

languages = (
    lang_q[["language"]]
    .drop_duplicates()
)

panel = (
    languages.assign(key=1)
    .merge(
        quarters.assign(key=1),
        on="key"
    )
    .drop(columns="key")
)

lang_q = panel.merge(
    lang_q[
        [
            "year",
            "quarter",
            "language",
            "num_pushers"
        ]
    ],
    on=["year", "quarter", "language"],
    how="left"
)

lang_q["num_pushers"] = (
    lang_q["num_pushers"]
    .fillna(0)
)





In [6]:

# ============================================================
# 6. CALCULATE QUARTERLY LANGUAGE ACTIVITY SHARE
#
# 중요:
# denominator는 FOCAL_LANGUAGES가 아니라
# dataset 내 전체 programming languages.
#
# 따라서:
#
# share_lt =
# num_pushers_lt /
# sum(num_pushers_jt)
#
# GitHub 전체 활동량 증가를 어느 정도 제거하기 위함.
# ============================================================

lang_q["total_language_activity"] = (
    lang_q.groupby(
        ["year", "quarter"]
    )["num_pushers"]
    .transform("sum")
)

lang_q["share"] = (
    lang_q["num_pushers"]
    / lang_q["total_language_activity"]
)

In [7]:
lang_q

,language,year,quarter,time,period,num_pushers,total_language_activity,share
0,1C Enterprise,2020,1,0,2020Q1,667.0,9848504.0,0.000068
1,1C Enterprise,2020,2,1,2020Q2,634.0,11213189.0,0.000057
2,1C Enterprise,2020,3,2,2020Q3,445.0,10508739.0,0.000042
3,1C Enterprise,2020,4,3,2020Q4,436.0,11429603.0,0.000038
4,1C Enterprise,2021,1,4,2021Q1,441.0,11547410.0,0.000038
...,...,...,...,...,...,...,...,...
8770,TXL,2025,1,20,2025Q1,0.0,20068128.0,0.000000
8771,TXL,2025,2,21,2025Q2,0.0,22637616.0,0.000000
8772,TXL,2025,3,22,2025Q3,0.0,22644092.0,0.000000
8773,TXL,2025,4,23,2025Q4,0.0,25795212.0,0.000000


In [8]:

# ============================================================
# 7. OPTIONAL: SELECT FOCAL LANGUAGES
#
# 반드시 share 계산 "후"에 필터링
# ============================================================

if FOCAL_LANGUAGES is not None:

    analysis_df = lang_q.loc[
        lang_q["language"].isin(FOCAL_LANGUAGES)
    ].copy()

else:

    analysis_df = lang_q.copy()



In [9]:
analysis_df

,language,year,quarter,time,period,num_pushers,total_language_activity,share
300,Assembly,2020,1,0,2020Q1,75836.0,9848504.0,0.007700
301,Assembly,2020,2,1,2020Q2,82361.0,11213189.0,0.007345
302,Assembly,2020,3,2,2020Q3,77001.0,10508739.0,0.007327
303,Assembly,2020,4,3,2020Q4,86670.0,11429603.0,0.007583
304,Assembly,2021,1,4,2021Q1,84292.0,11547410.0,0.007300
...,...,...,...,...,...,...,...,...
5270,Visual Basic .NET,2025,1,20,2025Q1,7184.0,20068128.0,0.000358
5271,Visual Basic .NET,2025,2,21,2025Q2,7663.0,22637616.0,0.000339
5272,Visual Basic .NET,2025,3,22,2025Q3,7665.0,22644092.0,0.000338
5273,Visual Basic .NET,2025,4,23,2025Q4,7978.0,25795212.0,0.000309


In [10]:
# ============================================================
# 8. FUNCTION:
#    CALCULATE BACKWARD-LOOKING ROLLING TREND
#
# 각 시점 t에서 과거 window개 분기만 이용:
#
# share_lt = alpha + beta * time
#
# relative slope:
#
# beta / mean(share)
#
# 예:
# relative_slope = 0.02
# -> 평균 share 대비 분기당 약 2% 증가하는 추세
#
# 미래 데이터는 절대 사용하지 않음.
# ============================================================

def calculate_rolling_trend(group, window=8):

    group = (
        group
        .sort_values("time")
        .copy()
    )

    group["slope"] = np.nan
    group["relative_slope"] = np.nan
    group["trend_pvalue"] = np.nan
    group["window_start"] = np.nan
    group["window_n"] = np.nan

    for i in range(window - 1, len(group)):

        temp = group.iloc[
            i - window + 1 : i + 1
        ].copy()

        # 실제 quarter index 사용
        x = temp["time"].values
        y = temp["share"].values

        X = sm.add_constant(x)

        model = sm.OLS(
            y,
            X
        ).fit()

        slope = model.params[1]
        pvalue = model.pvalues[1]

        mean_share = temp["share"].mean()

        if mean_share > 0:

            relative_slope = (
                slope / mean_share
            )

        else:

            relative_slope = np.nan

        idx = group.index[i]

        group.loc[idx, "slope"] = slope
        group.loc[idx, "relative_slope"] = relative_slope
        group.loc[idx, "trend_pvalue"] = pvalue
        group.loc[idx, "window_start"] = temp["time"].min()
        group.loc[idx, "window_n"] = len(temp)

    return group


# ============================================================
# 9. FUNCTION:
#    CLASSIFY RISING / STABLE / DECLINING
#
# Main:
# annual threshold = ±5%
#
# 이를 quarterly equivalent threshold로 변환:
#
# (1 + annual_threshold)^(1/4) - 1
#
# 약 ±1.23% per quarter
#
# p-value는 그룹 정의에 사용하지 않음.
# 이유:
# rolling window가 짧아서 p-value 기반 classification이
# sample size에 지나치게 민감할 수 있기 때문.
#
# p-value는 diagnostic 용도로 저장.
# ============================================================

def classify_trend(
    relative_slope,
    annual_threshold=0.05
):

    if pd.isna(relative_slope):
        return np.nan

    positive_threshold = (
        (1 + annual_threshold) ** (1 / 4)
        - 1
    )

    # 감소의 경우도 동일한 절대 threshold를 사용
    negative_threshold = -positive_threshold

    if relative_slope > positive_threshold:

        return "Rising"

    elif relative_slope < negative_threshold:

        return "Declining"

    else:

        return "Stable"



In [11]:
trend_list = []

for language, group in analysis_df.groupby("language"):

    # calculate_rolling_trend에 language 컬럼이 없어도 상관없게 함
    temp = calculate_rolling_trend(
        group.drop(columns=["language"], errors="ignore"),
        window=MAIN_WINDOW
    ).copy()

    # language를 명시적으로 다시 추가
    temp.insert(
        0,
        "language",
        language
    )

    trend_list.append(temp)


trend_main = pd.concat(
    trend_list,
    ignore_index=True
)


trend_main["trend_group"] = (
    trend_main["relative_slope"]
    .apply(
        classify_trend,
        annual_threshold=MAIN_ANNUAL_THRESHOLD
    )
)

In [12]:
trend_main

,language,year,quarter,time,period,num_pushers,total_language_activity,share,slope,relative_slope,trend_pvalue,window_start,window_n,trend_group
0,Assembly,2020,1,0,2020Q1,75836.0,9848504.0,0.007700,NaN,NaN,NaN,NaN,NaN,NaN
1,Assembly,2020,2,1,2020Q2,82361.0,11213189.0,0.007345,NaN,NaN,NaN,NaN,NaN,NaN
2,Assembly,2020,3,2,2020Q3,77001.0,10508739.0,0.007327,NaN,NaN,NaN,NaN,NaN,NaN
3,Assembly,2020,4,3,2020Q4,86670.0,11429603.0,0.007583,NaN,NaN,NaN,NaN,NaN,NaN
4,Assembly,2021,1,4,2021Q1,84292.0,11547410.0,0.007300,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720,Visual Basic .NET,2025,1,20,2025Q1,7184.0,20068128.0,0.000358,-0.000018,-0.040202,0.012741,13.0,8.0,Declining
721,Visual Basic .NET,2025,2,21,2025Q2,7663.0,22637616.0,0.000339,-0.000023,-0.054334,0.002354,14.0,8.0,Declining
722,Visual Basic .NET,2025,3,22,2025Q3,7665.0,22644092.0,0.000338,-0.000027,-0.064536,0.000238,15.0,8.0,Declining
723,Visual Basic .NET,2025,4,23,2025Q4,7978.0,25795212.0,0.000309,-0.000024,-0.062013,0.000356,16.0,8.0,Declining


In [13]:

# ============================================================
# 11. APPROXIMATE ANNUALIZED TREND
#
# 결과 해석을 편하게 하기 위한 변수.
#
# relative_slope가 작은 값이라는 가정하에서
# quarterly trend × 4를 annual percentage로 표현.
#
# classification에는 사용하지 않음.
# ============================================================

trend_main["annualized_trend_approx"] = (
    trend_main["relative_slope"] * 4
)


# ============================================================
# 12. FINAL MAIN RESULT
# ============================================================

result = trend_main[
    [
        "language",
        "year",
        "quarter",
        "period",
        "num_pushers",
        "share",
        "slope",
        "relative_slope",
        "annualized_trend_approx",
        "trend_pvalue",
        "trend_group"
    ]
].copy()

result = result.sort_values(
    ["time"]
    if "time" in result.columns
    else ["year", "quarter", "language"]
)

In [14]:
result[result['year'] == 2022]

,language,year,quarter,period,num_pushers,share,slope,relative_slope,annualized_trend_approx,trend_pvalue,trend_group
8,Assembly,2022,1,2022Q1,94431.0,0.007134,-0.000026,-0.003604,-0.014415,0.221522,Stable
33,C,2022,1,2022Q1,473157.0,0.035747,0.000217,0.006233,0.024933,0.089296,Stable
58,C#,2022,1,2022Q1,363812.0,0.027486,-0.000029,-0.001046,-0.004186,0.844959,Stable
83,C++,2022,1,2022Q1,486903.0,0.036785,-0.000435,-0.011408,-0.045630,0.050625,Stable
108,Dart,2022,1,2022Q1,85814.0,0.006483,0.000259,0.043341,0.173366,0.000839,Rising
...,...,...,...,...,...,...,...,...,...,...,...
611,Scala,2022,4,2022Q4,35223.0,0.002309,-0.000076,-0.031340,-0.125358,0.006495,Declining
636,Solidity,2022,4,2022Q4,45962.0,0.003013,0.000277,0.106515,0.426061,0.003962,Rising
661,Swift,2022,4,2022Q4,166075.0,0.010888,0.000103,0.009647,0.038587,0.040287,Stable
686,TypeScript,2022,4,2022Q4,744070.0,0.048781,0.001707,0.038870,0.155480,0.000012,Rising


In [21]:


# ============================================================
# 13. PRE-GPT STATUS
#
# ChatGPT 이전 마지막 완전한 quarter = 2022Q3
#
# 이 시점의 그룹은 오직:
#
# 2020Q4 ~ 2022Q3
#
# 과거 8분기 데이터로 만들어짐.
# GPT 이후 정보는 전혀 들어가지 않음.
# ============================================================

pre_gpt_status = (
    result.loc[
        (result["year"] == PRE_GPT_YEAR)
        & (result["quarter"] == PRE_GPT_QUARTER),
        [
            "language",
            "share",
            "relative_slope",
            "annualized_trend_approx",
            "trend_pvalue",
            "trend_group"
        ]
    ]
    .sort_values(
        "relative_slope"
    )
    .reset_index(drop=True)
)

def get_key_by_value(d, target_value):
    return next(
        (k for k, v in d.items() if v == target_value),
        None
    )


# 'python'

pre_gpt_status['language'] = pre_gpt_status['language'].apply(lambda x: get_key_by_value(CONSTANTS.language_map, x))

print("\n==============================")
print("PRE-GPT LANGUAGE STATUS")
print("==============================")

pre_gpt_status.to_csv('./pre_gpt_status.csv', index=False)

pre_gpt_status



PRE-GPT LANGUAGE STATUS


,language,share,relative_slope,annualized_trend_approx,trend_pvalue,trend_group
0,ruby,0.023648,-0.078655,-0.314621,0.000101,Declining
1,objective-c,0.012575,-0.059906,-0.239623,0.000658,Declining
2,f#,0.000403,-0.056324,-0.225296,0.110153,Declining
3,scala,0.002155,-0.049552,-0.198206,0.001425,Declining
4,vb.net,0.000471,-0.038677,-0.154706,0.130104,Declining
5,matlab,0.003339,-0.037129,-0.148515,0.001929,Declining
6,fortran,0.001508,-0.036958,-0.147832,0.000843,Declining
7,haskell,0.001346,-0.036751,-0.147003,0.164316,Declining
8,r,0.006450,-0.032321,-0.129285,0.000038,Declining
9,groovy,0.002702,-0.030240,-0.120960,0.002762,Declining
